In [1]:
!pip install -q ultralytics opencv-python

In [2]:

from google.colab import files

uploaded = files.upload()


Saving input.mp4 to input.mp4


In [3]:

from ultralytics import YOLO
import cv2
import os

# Load YOLO11 nano model
model = YOLO("yolo11n.pt")

print("YOLO11 loaded successfully!")


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
YOLO11 loaded successfully!


In [4]:

# Input video
video_path = "input.mp4"

# Open video
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error opening video!")
else:
    print("Video opened successfully!")

# Get video properties
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

# Create output folder
os.makedirs("outputs", exist_ok=True)

# Output video
output_path = "outputs/detection_output.mp4"

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    output_path,
    fourcc,
    fps,
    (width, height)
)

print("Starting person detection...")

# Process each frame
while True:

    ret, frame = cap.read()

    if not ret:
        break

    # Detect people only
    results = model.predict(
        frame,
        classes=[0],
        conf=0.35,
        verbose=False
    )

    # Draw bounding boxes
    annotated_frame = results[0].plot()

    # Save frame
    out.write(annotated_frame)

# Release resources
cap.release()
out.release()

print("Detection completed!")
print("Output:", output_path)


Video opened successfully!
Starting person detection...
Detection completed!
Output: outputs/detection_output.mp4


In [5]:

from google.colab import files

files.download("outputs/detection_output.mp4")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
from ultralytics import YOLO

# Load YOLO11 nano model
model = YOLO("yolo11n.pt")

print("YOLO11 tracking model loaded successfully!")

YOLO11 tracking model loaded successfully!


In [7]:

import cv2
import os

# Input video
video_path = "input.mp4"

# Open video
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise Exception("Could not open input video!")

# Get video properties
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

# Create output folder
os.makedirs("outputs", exist_ok=True)

# Output video
output_path = "outputs/tracking_output.mp4"

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    output_path,
    fourcc,
    fps,
    (width, height)
)

# Store unique tracking IDs
unique_ids = set()

print("Starting ByteTrack tracking...")

# Process video frame by frame
while True:

    ret, frame = cap.read()

    if not ret:
        break

    # Detect and track people
    results = model.track(
        frame,
        persist=True,
        classes=[0],
        conf=0.35,
        tracker="bytetrack.yaml",
        verbose=False
    )

    # Get current frame results
    result = results[0]

    # Collect tracking IDs
    if result.boxes.id is not None:

        ids = result.boxes.id.int().cpu().tolist()

        for track_id in ids:
            unique_ids.add(track_id)

    # Draw boxes and tracking IDs
    annotated_frame = result.plot()

    # Save processed frame
    out.write(annotated_frame)

# Release resources
cap.release()
out.release()

print("Tracking completed!")
print("Unique tracking IDs:", len(unique_ids))
print("Output saved:", output_path)


Starting ByteTrack tracking...
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.13.16 environment at: /usr
Resolved 2 packages in 168ms
Prepared 1 package in 27ms
Installed 1 package in 1ms
 + lap==0.5.14

requirements: AutoUpdate success ✅ 0.5s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

Tracking completed!
Unique tracking IDs: 132
Output saved: outputs/tracking_output.mp4


In [8]:
from google.colab import files

files.download("outputs/tracking_output.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:

import cv2
import os
import numpy as np
from ultralytics import YOLO

# Load YOLO model
model = YOLO("yolo11n.pt")

# Open input video
cap = cv2.VideoCapture("input.mp4")

if not cap.isOpened():
    raise Exception("Could not open video!")

# Video properties
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

os.makedirs("outputs", exist_ok=True)

# Final output video
output_path = "outputs/final_output.mp4"

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    output_path, fourcc, fps, (width, height)
)

# Virtual counting line
LINE_X = width // 2

# Define restricted zone polygon
restricted_zone = np.array([
    [int(width * 0.40), int(height * 0.30)],
    [int(width * 0.70), int(height * 0.30)],
    [int(width * 0.70), int(height * 0.75)],
    [int(width * 0.40), int(height * 0.75)]
], dtype=np.int32)

# Counters
entry_count = 0
exit_count = 0

# Store previous positions
previous_positions = {}

# Store IDs already counted
counted_ids = set()

# Store IDs that have already triggered an alert
alerted_ids = set()

print("Starting people monitoring system...")

while True:

    ret, frame = cap.read()

    if not ret:
        break

    # Track people
    results = model.track(
        frame,
        persist=True,
        classes=[0],
        conf=0.35,
        tracker="bytetrack.yaml",
        verbose=False
    )

    result = results[0]

    # Draw virtual counting line
    cv2.line(
        frame,
        (LINE_X, 0),
        (LINE_X, height),
        (0, 255, 255),
        2
    )

    # Draw restricted zone
    overlay = frame.copy()

    cv2.fillPoly(
        overlay,
        [restricted_zone],
        (0, 0, 255)
    )

    frame = cv2.addWeighted(
        overlay, 0.20, frame, 0.80, 0
    )

    cv2.polylines(
        frame,
        [restricted_zone],
        True,
        (0, 0, 255),
        2
    )

    cv2.putText(
        frame,
        "RESTRICTED ZONE",
        (restricted_zone[0][0], restricted_zone[0][1] - 10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (0, 0, 255),
        2
    )

    current_inside = set()

    # Process tracked people
    if result.boxes.id is not None:

        boxes = result.boxes.xyxy.cpu().numpy()
        ids = result.boxes.id.int().cpu().tolist()

        for box, track_id in zip(boxes, ids):

            x1, y1, x2, y2 = map(int, box)

            # Calculate center point
            center_x = (x1 + x2) // 2
            center_y = (y1 + y2) // 2

            # Draw bounding box
            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                2
            )

            # Display tracking ID
            cv2.putText(
                frame,
                f"ID: {track_id}",
                (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (255, 255, 0),
                2
            )

            # Draw center point
            cv2.circle(
                frame,
                (center_x, center_y),
                4,
                (255, 0, 0),
                -1
            )

            # ENTRY AND EXIT COUNTING
            if track_id in previous_positions:

                previous_x = previous_positions[track_id]

                if (
                    previous_x < LINE_X
                    and center_x >= LINE_X
                    and track_id not in counted_ids
                ):
                    entry_count += 1
                    counted_ids.add(track_id)

                elif (
                    previous_x > LINE_X
                    and center_x <= LINE_X
                    and track_id not in counted_ids
                ):
                    exit_count += 1
                    counted_ids.add(track_id)

            previous_positions[track_id] = center_x

            # RESTRICTED ZONE DETECTION
            inside = cv2.pointPolygonTest(
                restricted_zone,
                (float(center_x), float(center_y)),
                False
            )

            if inside >= 0:

                current_inside.add(track_id)

                # Change bounding box to red
                cv2.rectangle(
                    frame,
                    (x1, y1),
                    (x2, y2),
                    (0, 0, 255),
                    3
                )
# Alert each tracking ID only once
if track_id not in alerted_ids:

    print(
        f"ALERT! Person ID {track_id} "
        "entered restricted zone!"
    )

    alerted_ids.add(track_id)

    # Display counts
    cv2.putText(
        frame,
        f"Entries: {entry_count}",
        (20, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 0),
        2
    )

    cv2.putText(
        frame,
        f"Exits: {exit_count}",
        (20, 70),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 150, 0),
        2
    )

    # Display intrusion alert
    if current_inside:

        cv2.putText(
            frame,
            "WARNING: RESTRICTED ZONE INTRUSION!",
            (20, height - 20),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (0, 0, 255),
            2
        )


    # Save final frame
    out.write(frame)

# Release resources
cap.release()
out.release()

print("\nMonitoring completed!")
print("Total Entries:", entry_count)
print("Total Exits:", exit_count)
print("Output saved:", output_path)


Starting people monitoring system...
ALERT! Person ID 238 entered restricted zone!

Monitoring completed!
Total Entries: 16
Total Exits: 10
Output saved: outputs/final_output.mp4


In [10]:
from google.colab import files

files.download("outputs/people_counting.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
from google.colab import files

files.download("outputs/final_output.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>